In [1]:
# =========================================
# CHILD JUMPING TEST (DATA-DRIVEN)
# =========================================

import cv2
import numpy as np
import mediapipe as mp
from scipy.signal import find_peaks
from ultralytics import YOLO

In [2]:
# -------------------------------
# MediaPipe Setup
# -------------------------------
mpPose = mp.solutions.pose
pose = mpPose.Pose(min_detection_confidence=0.6)

mpDraw = mp.solutions.drawing_utils
DRAW_LM = mpDraw.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=2)
DRAW_CONN = mpDraw.DrawingSpec(color=(0,0,0), thickness=2)

In [3]:
# -------------------------------
# Utils
# -------------------------------
def safe_mean(x): return float(np.mean(x)) if len(x) else 0.0
def safe_std(x): return float(np.std(x)) if len(x) else 0.0

def smooth(x, k=5):
    if len(x) < k: return np.array(x)
    return np.convolve(x, np.ones(k)/k, mode='same')

def angle(a,b,c):
    a,b,c = np.array(a),np.array(b),np.array(c)
    ba, bc = a-b, c-b
    cos = np.dot(ba,bc)/(np.linalg.norm(ba)*np.linalg.norm(bc)+1e-6)
    return np.degrees(np.arccos(np.clip(cos,-1,1)))

In [4]:
def torso_angle(l_sh, r_sh, l_hip, r_hip):
    sh_mid = np.array([(l_sh[0]+r_sh[0])/2, (l_sh[1]+r_sh[1])/2])
    hip_mid = np.array([(l_hip[0]+r_hip[0])/2, (l_hip[1]+r_hip[1])/2])
    vec = sh_mid - hip_mid
    vertical = np.array([0, -1.0])  # image coords (y down)
    cos = np.dot(vec, vertical) / (np.linalg.norm(vec)*np.linalg.norm(vertical)+1e-6)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

In [5]:
# Two-Foot Takeoff
def takeoff_score_cal(takeoff_ratio):
    takeoff_score = 0
    if takeoff_ratio > 0.85: takeoff_score = 5
    elif takeoff_ratio > 0.7: takeoff_score = 4
    elif takeoff_ratio > 0.5: takeoff_score = 3
    elif takeoff_ratio > 0.3: takeoff_score = 2
    else: takeoff_score = 1
    return takeoff_score

In [6]:
# Two-Foot Landing
def landing_score_cal(landing_ratio):
    landing_score = 0
    if landing_ratio > 0.85: landing_score = 5
    elif landing_ratio > 0.7: landing_score = 4
    elif landing_ratio > 0.5: landing_score = 3
    elif landing_ratio > 0.3: landing_score = 2
    else: landing_score = 1
    return landing_score

In [7]:
# Arm Swing
def arm_score_cal(arm_amp):
    arm_score = 0
    if arm_amp > 0.12: arm_score = 5
    elif arm_amp > 0.09: arm_score = 4
    elif arm_amp > 0.06: arm_score = 3
    elif arm_amp > 0.03: arm_score = 2
    else: arm_score = 1
    return arm_score

In [8]:
# Body Alignment (lower angle deviation = better)
def align_score_cal(align_mean):
    align_score = 0
    if align_mean < 10: align_score = 5
    elif align_mean < 20: align_score = 4
    elif align_mean < 30: align_score = 3
    elif align_mean < 40: align_score = 2
    else: align_score = 1
    return align_score

In [9]:
def threshold_line(path):
    
    model = YOLO("best.pt")
    cap = cv2.VideoCapture(path)

    left_x, right_x = [], []
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, conf=0.5, verbose=False)

        centers = []
        for box in results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf.item()
            cx = (x1 + x2) // 2
            centers.append(cx)
            
            # Draw rectangle on cone
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Optional: confidence label
            cv2.putText(frame, f"Cone {conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        if len(centers) >= 2:
            centers.sort()
            left_x.append(centers[0])
            right_x.append(centers[-1])

        if len(left_x) > 50:
            break

        cv2.namedWindow("Video", cv2.WINDOW_NORMAL)
        cv2.setWindowProperty("Video", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
        cv2.imshow("Video", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    # Final fixed threshold
    left_avg = int(np.mean(left_x))
    right_avg = int(np.mean(right_x))
    # threshold_x = int((left_avg + right_avg) / 2)

    print("Left cone X:", left_avg)
    print("Right cone X:", right_avg)
    # print("Threshold X:", threshold_x)
    return left_avg, right_avg

In [12]:
# MAIN FUNCTION
# -------------------------------
def jumping_test(path="video.mp4"):

    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    #threshold line calculate  
    left_line_x, right_line_x = 0.10, 0.85
    left_line_x, right_line_x = threshold_line(path)
    left_line_x = round((left_line_x/frame_width), 2)
    right_line_x = round((right_line_x/frame_width), 2)
    print(f"{left_line_x} --- {right_line_x}")

    # signals
    hip_y = []
    l_ank_y, r_ank_y = [], []
    l_el_y, r_el_y = [], []
    l_wri_y, r_wri_y = [], []
    l_sh_y, r_sh_y = [], []
    torso_ang = []

    frame_idx = 0
    window = "Jumping Analysis"
    cv2.namedWindow(window, cv2.WINDOW_NORMAL)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = pose.process(rgb)

        if res.pose_landmarks:
            lms = res.pose_landmarks.landmark

            # hip center
            hx = (lms[23].x + lms[24].x) / 2

            # boundary check
            if not (left_line_x <= hx <= right_line_x):
                continue

            # visibility filter
            if lms[23].visibility < 0.5:
                cv2.imshow(window, frame)
                if cv2.waitKey(1) & 0xFF == 27: break
                frame_idx += 1
                continue

            mpDraw.draw_landmarks(frame, res.pose_landmarks, mpPose.POSE_CONNECTIONS, DRAW_LM, DRAW_CONN)

            # keypoints
            l_sh=(lms[11].x,lms[11].y); r_sh=(lms[12].x,lms[12].y)
            l_el=(lms[13].x,lms[13].y); r_el=(lms[14].x,lms[14].y)
            l_wri=(lms[15].x,lms[15].y); r_wri=(lms[16].x,lms[16].y)
            l_hip=(lms[23].x,lms[23].y); r_hip=(lms[24].x,lms[24].y)
            l_ank=(lms[27].x,lms[27].y); r_ank=(lms[28].x,lms[28].y)

            # COM (hip)
            hy = (l_hip[1] + r_hip[1]) / 2
            hip_y.append(hy)

            # ankles
            l_ank_y.append(l_ank[1]); r_ank_y.append(r_ank[1])

            # arms
            l_el_y.append(l_el[1]); r_el_y.append(r_el[1])
            l_wri_y.append(l_wri[1]); r_wri_y.append(r_wri[1])
            l_sh_y.append(l_sh[1]); r_sh_y.append(r_sh[1])

            # torso alignment
            torso_ang.append(torso_angle(l_sh, r_sh, l_hip, r_hip))

        # display
        cv2.putText(frame, f"Frame: {frame_idx}", (20,30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
        cv2.imshow(window, frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

        frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()

    # -------------------------------
    # PREPROCESS
    # -------------------------------
    hip_y = smooth(hip_y)
    l_ank_y = smooth(l_ank_y)
    r_ank_y = smooth(r_ank_y)

    n = min(len(hip_y), len(l_ank_y), len(r_ank_y))
    if n < 12:
        return 1,1,1,1

    hip_y, l_ank_y, r_ank_y = hip_y[:n], l_ank_y[:n], r_ank_y[:n]

    # -------------------------------
    # JUMP DETECTION (COM peaks)
    # -------------------------------
    peaks, _ = find_peaks(-hip_y, distance=6)
    peaks = peaks[peaks < n]

    takeoff_diffs = []
    landing_diffs = []
    arm_patterns = []
    align_vals = []

    for p in peaks:
        # local window
        i0 = max(p-3, 0)
        i1 = min(p+3, n-1)

        # velocities
        vel_l = np.diff(l_ank_y[i0:i1+1])
        vel_r = np.diff(r_ank_y[i0:i1+1])

        # takeoff: max upward velocity (negative y direction)
        if len(vel_l) and len(vel_r):
            t_l = np.argmin(vel_l)
            t_r = np.argmin(vel_r)
            takeoff_diffs.append(abs(t_l - t_r))

            # landing: max downward velocity
            l_l = np.argmax(vel_l)
            l_r = np.argmax(vel_r)
            landing_diffs.append(abs(l_l - l_r))

        # arm swing (back → forward)
        # wrist relative to shoulder over window
        wr = np.array(l_wri_y[i0:i1+1]) - np.array(l_sh_y[i0:i1+1])
        if len(wr) > 3:
            arm_patterns.append(np.max(wr) - np.min(wr))

        # alignment
        align_vals.append(safe_mean(torso_ang[i0:i1+1]))

    # -------------------------------
    # METRICS
    # -------------------------------
    takeoff_sync = safe_mean(takeoff_diffs)
    landing_sync = safe_mean(landing_diffs)
    arm_amp = safe_mean(arm_patterns)
    align_mean = safe_mean(align_vals)

    # normalize sync by window size
    # smaller diff = better simultaneity
    # convert to ratio
    takeoff_ratio = 1 - (takeoff_sync / 5.0)
    landing_ratio = 1 - (landing_sync / 5.0)

    # =========================================================
    # SCORING
    # =========================================================

    # Two-Foot Takeoff
    takeoff_score = takeoff_score_cal(takeoff_ratio)

    # Two-Foot Landing
    landing_score = landing_score_cal(landing_ratio)

    # Arm Swing
    arm_score = arm_score_cal(arm_amp)

    # Body Alignment (lower angle deviation = better)
    align_score = align_score_cal(align_mean)

    print("------ JUMPING RESULT ------")
    print("Takeoff:", takeoff_score)
    print("Landing:", landing_score)
    print("Arm Swing:", arm_score)
    print("Alignment:", align_score)

    final_score = (takeoff_score + landing_score + arm_score + align_score)/4

    return final_score

In [13]:
path = "data/jumpping.mp4"
print(jumping_test(path))

Left cone X: 65
Right cone X: 688
0.08 --- 0.81
------ JUMPING RESULT ------
Takeoff: 4
Landing: 4
Arm Swing: 2
Alignment: 5
3.75
